# Day 13 - Merging, Concatenating and Reshaping  (SOLUTIONS)

**Goal:** Combine tables and switch between wide and long formats.

**Quick recap (read for 2 minutes):**

- `df.merge(other, on='key', how='left')` joins tables (like VLOOKUP / SQL JOIN)
- `pd.concat([a, b])` stacks rows; `axis=1` places side by side
- `pd.melt(df, id_vars=[...], value_vars=[...])` wide -> long
- `df.pivot(index=, columns=, values=)` long -> wide
- `.unstack()` moves an index level into columns
- `indicator=True` in merge shows which rows matched

*There are many valid ways to solve each question. If your answer gives the same result, it is correct.*

### Warm-up (5 min)
Without looking, redo Questions 1 to 3 from **Day 12** in a scratch cell below.

### Setup (run this first)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("spotify_user_behv.csv")

### Q1. Create a small DataFrame `region_df` mapping each country to a region (Asia, Europe, North America, South America, Oceania). Build it from a dictionary.

In [ ]:
region_df = pd.DataFrame({
    "country": ["India", "Indonesia", "Germany", "Spain", "UK", "Italy", "France", "USA", "Canada", "Mexico", "Brazil", "Australia"],
    "region": ["Asia", "Asia", "Europe", "Europe", "Europe", "Europe", "Europe", "North America", "North America", "North America", "South America", "Oceania"],
})
region_df

### Q2. Left-merge `region_df` into `df` and count users per region.

In [ ]:
region_df = pd.DataFrame({"country": ["India", "Indonesia", "Germany", "Spain", "UK", "Italy", "France", "USA", "Canada", "Mexico", "Brazil", "Australia"], "region": ["Asia", "Asia", "Europe", "Europe", "Europe", "Europe", "Europe", "North America", "North America", "North America", "South America", "Oceania"]})
merged = df.merge(region_df, on="country", how="left")
merged["region"].value_counts()

### Q3. Verify the merge did not add or lose rows (use `assert`).

In [ ]:
assert len(merged) == len(df)
print("Row count unchanged:", len(merged))

### Q4. Build a price table (Free 0, Student 5.99, Premium Individual 11.99, Premium Duo 16.99, Premium Family 19.99; illustrative prices), merge it, and compute total monthly revenue per country from Active users.

In [ ]:
price = pd.DataFrame({"subscription_type": ["Free", "Student", "Premium Individual", "Premium Duo", "Premium Family"], "monthly_price_usd": [0, 5.99, 11.99, 16.99, 19.99]})
m = df.merge(price, on="subscription_type", how="left")
m[m["subscription_status"] == "Active"].groupby("country")["monthly_price_usd"].sum().round(0).sort_values(ascending=False)

### Q5. Split `df` into its first 100 rows (`a`) and next 100 rows (`b`), then stack them with `pd.concat` and a clean index.

In [ ]:
a = df.iloc[:100]
b = df.iloc[100:200]
stacked = pd.concat([a, b], ignore_index=True)
stacked.shape

### Q6. Place `user_id`+`age` (first 5 rows) side by side with `country` (first 5 rows) using `concat` with `axis=1`.

In [ ]:
pd.concat([df[["user_id", "age"]].head(5), df[["country"]].head(5)], axis=1)

### Q7. Use `melt` to turn hours, skips and playlists of the first 5 users into long format (columns: user_id, metric, value).

In [ ]:
pd.melt(df.head(5), id_vars=["user_id"], value_vars=["avg_listening_hours_per_week", "avg_skips_per_day", "playlists_created"], var_name="metric", value_name="value")

### Q8. Compute average hours by country and device, then `pivot` it into a country x device table.

In [ ]:
g = df.groupby(["country", "primary_device"])["avg_listening_hours_per_week"].mean().reset_index()
g.pivot(index="country", columns="primary_device", values="avg_listening_hours_per_week").round(2)

### Q9. Count users by country and subscription type, then `unstack` so subscription types become columns.

In [ ]:
df.groupby(["country", "subscription_type"]).size().unstack(fill_value=0)

### Q10. Merge with a lookup table that is missing India, using `how='left'` and `indicator=True`. How many rows did not match?

In [ ]:
lookup = pd.DataFrame({"country": [c for c in df["country"].unique() if c != "India"], "flag": 1})
chk = df.merge(lookup, on="country", how="left", indicator=True)
chk["_merge"].value_counts()